In [14]:
from openfermion import InteractionOperator, jordan_wigner
from openfermion.transforms import get_fermion_operator

for i, frag_data in enumerate(dmet.scf_fragments):
    h1e   = frag_data[1]
    h2e   = frag_data[5]
    n_elec = int(sum(frag_data[3]))

    # Build and convert — h2e is in chemist notation (ij|kl)
    mol_ham    = InteractionOperator(0.0, h1e, 0.5 * h2e)
    f_ham      = get_fermion_operator(mol_ham)
    q_ham      = jordan_wigner(f_ham)
    print(f"Fragment {i} qubit Hamiltonian:\n{q_ham}")

Fragment 0 qubit Hamiltonian:
(-2.951083908629057+0j) [] +
(-0.23195682263075973+0j) [X0 X1] +
(0.03239001447592098+0j) [X0 Z1 X2] +
(-0.058452638969648286+0j) [X0 Z1 Z2 X3] +
(-0.23195682263075973+0j) [Y0 Y1] +
(0.03239001447592098+0j) [Y0 Z1 Y2] +
(-0.058452638969648286+0j) [Y0 Z1 Z2 Y3] +
(0.6870476698720374+0j) [Z0] +
(-0.13386538311870444+0j) [X1 X2] +
(0.23709998045364128+0j) [X1 Z2 X3] +
(-0.13386538311870444+0j) [Y1 Y2] +
(0.23709998045364128+0j) [Y1 Z2 Y3] +
(0.7884942844424874+0j) [Z1] +
(0.06509004814137803+0j) [X2 X3] +
(0.06509004814137803+0j) [Y2 Y3] +
(0.9661130606632704+0j) [Z2] +
(0.5094288936512621+0j) [Z3]
Fragment 1 qubit Hamiltonian:
(-2.9510839086290583+0j) [] +
(-0.23195682263076128+0j) [X0 X1] +
(0.13386538311870386+0j) [X0 Z1 X2] +
(0.23709998045364195+0j) [X0 Z1 Z2 X3] +
(-0.23195682263076128+0j) [Y0 Y1] +
(0.13386538311870386+0j) [Y0 Z1 Y2] +
(0.23709998045364195+0j) [Y0 Z1 Z2 Y3] +
(0.7884942844424914+0j) [Z0] +
(-0.03239001447592046+0j) [X1 X2] +
(-0.058452

In [ ]:
from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition
from openfermion import InteractionOperator, jordan_wigner
from openfermion.transforms import get_fermion_operator

# ── 1. Setup & Run ────────────────────────────────────────────────────────
geometry = [
    ("H", (0., 0., 0.00)),
    ("H", (0., 0., 0.74)),
    ("H", (0., 0., 1.48)),
    ("H", (0., 0., 2.22)),
]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
energy = dmet.simulate()
print(f"DMET energy = {energy:.8f} Ha")

# ── 2. Extract Hamiltonians directly from known structure ─────────────────
# scf_fragments[i] = [RHF, h1e(4x4), Mole, [n_a,n_b], fock(4x4), h2e(4x4x4x4), fock_copy(4x4)]
#                     [0]   [1]        [2]   [3]         [4]        [5]            [6]

qubit_hams      = []
fermionic_hams  = []

for i, frag_data in enumerate(dmet.scf_fragments):

    h1e            = frag_data[1]               # one-electron integrals (4×4)
    n_elec_list    = frag_data[3]               # [n_alpha, n_beta]
    h2e            = frag_data[5]               # two-electron integrals (4×4×4×4)

    n_active_mos   = h1e.shape[0]               # 4
    n_active_sos   = 2 * n_active_mos           # 8
    n_electrons    = int(sum(n_elec_list))       # 4

    print(f"\n{'='*50}")
    print(f"Fragment {i}")
    print(f"  n_electrons  : {n_electrons}")
    print(f"  n_active_mos : {n_active_mos}")
    print(f"  n_active_sos : {n_active_sos}")
    print(f"  h1e:\n{h1e}")
    print(f"  h2e shape    : {h2e.shape}")

    # ── FermionOperator via OpenFermion ───────────────────────────────────
    # InteractionOperator uses chemist notation: (ij|kl)
    # constant=0 since nuclear repulsion is handled at DMET level
    interaction_ham = InteractionOperator(
        constant              = 0.0,
        one_body_tensor       = h1e,
        two_body_tensor       = 0.5 * h2e,   # factor 0.5 from double counting
    )

    f_ham = get_fermion_operator(interaction_ham)
    fermionic_hams.append(f_ham)
    print(f"\n  Fermionic Hamiltonian:\n{f_ham}")

    # ── QubitOperator via Jordan-Wigner ───────────────────────────────────
    q_ham = jordan_wigner(f_ham)
    qubit_hams.append(q_ham)
    print(f"\n  Qubit Hamiltonian (JW):\n{q_ham}")

# ── 3. Summary ────────────────────────────────────────────────────────────
print("\n\nReady to use:")
print(f"  fermionic_hams[0] terms : {len(list(fermionic_hams[0].terms))}")
print(f"  fermionic_hams[1] terms : {len(list(fermionic_hams[1].terms))}")
print(f"  qubit_hams[0]     terms : {len(list(qubit_hams[0].terms))}")
print(f"  qubit_hams[1]     terms : {len(list(qubit_hams[1].terms))}")

In [17]:
"""
Pipeline: DMET (Tangelo) → Extract h1e/h2e → SQD (qiskit-addon-sqd)
"""

import numpy as np
from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition

from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit.primitives import StatevectorSampler

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ─────────────────────────────────────────────────────────────────────────────
# 1. DMET — get reference energy + embedded integrals
# ─────────────────────────────────────────────────────────────────────────────
geometry = [("H", (0., 0., i * 0.74)) for i in range(4)]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"DMET/FCI reference energy = {dmet_energy:.8f} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Extract fragment 0 integrals (same structure we confirmed earlier)
#    scf_fragments[i] = [RHF, h1e, Mole, [n_a,n_b], fock, h2e, fock_copy]
#                        [0]   [1]  [2]   [3]        [4]   [5]  [6]
# ─────────────────────────────────────────────────────────────────────────────
frag_data = dmet.scf_fragments[0]
h1e       = frag_data[1]                   # (4,4)  one-electron integrals
h2e       = frag_data[5]                   # (4,4,4,4) two-electron integrals
n_alpha   = int(frag_data[3][0])           # 2
n_beta    = int(frag_data[3][1])           # 2
n_orb     = h1e.shape[0]                   # 4
n_qubits  = 2 * n_orb                      # 8

print(f"\nFragment 0 → {n_orb} orbitals | {n_alpha}α + {n_beta}β electrons | {n_qubits} qubits")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Build quantum circuit
#    Convention (qiskit-addon-sqd):
#      qubits 0 … n_orb-1      → alpha spin-orbitals
#      qubits n_orb … 2*n_orb-1 → beta  spin-orbitals
#
#    Strategy: prepare HF state first, then apply EfficientSU2 ansatz
#    This ensures most samples are near the physically relevant sector
# ─────────────────────────────────────────────────────────────────────────────

# Hartree-Fock state preparation
hf_circuit = QuantumCircuit(n_qubits)
for i in range(n_alpha):          # fill lowest alpha orbitals
    hf_circuit.x(i)
for i in range(n_beta):           # fill lowest beta orbitals
    hf_circuit.x(n_orb + i)

# Hardware-efficient ansatz on top of HF state
ansatz = EfficientSU2(
    n_qubits,
    reps                      = 2,
    entanglement              = "linear",
    skip_final_rotation_layer = True,
)
rng    = np.random.default_rng(42)
params = rng.uniform(-0.3, 0.3, ansatz.num_parameters)   # small deviation from HF

# Full circuit = HF prep + ansatz + measurement
circuit = hf_circuit.compose(ansatz.assign_parameters(params))
circuit.measure_all()

print(f"\nCircuit: {circuit.num_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Sample from statevector simulator
# ─────────────────────────────────────────────────────────────────────────────
sampler = StatevectorSampler()
n_shots = 20_000

counts = (
    sampler
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)
print(f"Unique bitstrings sampled: {len(counts)}")

# counts_to_arrays returns:
#   bitstring_matrix : bool array (n_samples, n_qubits)
#   probs            : float array (n_samples,)
bitstring_matrix, probs = counts_to_arrays(counts)
print(f"Bitstring matrix shape: {bitstring_matrix.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. SQD self-consistent loop
#
#    Each iteration:
#      a) recover_configurations — flips bits to enforce correct n_alpha/n_beta
#                                   guided by avg orbital occupancies
#      b) solve_fermion          — diagonalises H in the bitstring subspace
#                                   returns energy + avg occupancies for next iter
# ─────────────────────────────────────────────────────────────────────────────
n_iterations = 6
sqd_energy   = None
avg_occs     = None         # shape (2*n_orb,) — updated each iteration

print("\nSQD iterations:")
print(f"{'─'*40}")

for it in range(n_iterations):

    # ── a) Configuration recovery (skip on iteration 0) ──────────────────
    if avg_occs is not None:
        bitstring_matrix, probs = recover_configurations(
            bitstring_matrix,
            probs,
            avg_occs,
            num_elec_a     = n_alpha,
            num_elec_b     = n_beta,
            num_iterations = 10,
            rand_seed      = 42,
        )

    if bitstring_matrix.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configurations — increase n_shots")
        break

    # ── b) Solve H in bitstring subspace ─────────────────────────────────
    #   solve_fermion returns:
    #     sqd_energy  : float        ground state energy
    #     coeffs      : ndarray      CI coefficients in subspace
    #     avg_occs    : ndarray      avg orbital occupancies → fed to next iter
    sqd_energy, coeffs, avg_occs = solve_fermion(
        bitstring_matrix,
        hcore      = h1e,          # one-electron integrals
        eri        = h2e,          # two-electron integrals
        num_alpha  = n_alpha,
        num_beta   = n_beta,
        open_shell = False,        # RHF fragment (same alpha/beta)
        spin_sq    = 0.0,          # target S²=0 (singlet)
    )

    print(f"  Iteration {it+1:02d}  |  E = {sqd_energy:.8f} Ha  |  {bitstring_matrix.shape[0]} configs")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Final comparison
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*45}")
print(f"  DMET/FCI reference  : {dmet_energy:.8f} Ha")
print(f"  SQD result (frag 0) : {sqd_energy:.8f} Ha")
print(f"  Δ energy            : {abs(sqd_energy - dmet_energy):.2e} Ha")
print(f"{'═'*45}")

DMET/FCI reference energy = -2.13888991 Ha

Fragment 0 → 4 orbitals | 2α + 2β electrons | 8 qubits

Circuit: 8 qubits | 3 depth | 32 params
Unique bitstrings sampled: 51
Bitstring matrix shape: (51, 8)

SQD iterations:
────────────────────────────────────────


/tmp/ipykernel_8056/4076095813.py:66: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(


ValueError: Spin-up CI string in index 0 has hamming weight 0, but CI string in index 1 has hamming weight 1.

In [18]:
import numpy as np
from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition

from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2          # ← function, not class
from qiskit.primitives import StatevectorSampler

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ─────────────────────────────────────────────────────────────────────────────
# 1. DMET
# ─────────────────────────────────────────────────────────────────────────────
geometry = [("H", (0., 0., i * 0.74)) for i in range(4)]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"DMET/FCI reference energy = {dmet_energy:.8f} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Extract fragment 0 integrals
# ─────────────────────────────────────────────────────────────────────────────
frag_data = dmet.scf_fragments[0]
h1e       = frag_data[1]
h2e       = frag_data[5]
n_alpha   = int(frag_data[3][0])    # 2
n_beta    = int(frag_data[3][1])    # 2
n_orb     = h1e.shape[0]            # 4
n_qubits  = 2 * n_orb               # 8

print(f"Fragment 0 → {n_orb} orbs | {n_alpha}α + {n_beta}β electrons | {n_qubits} qubits")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Helper: filter bitstrings to correct electron count
#    solve_fermion infers n_alpha/n_beta from Hamming weights
#    → ALL bitstrings must have exactly n_alpha ones in alpha sector
#       and n_beta ones in beta sector
# ─────────────────────────────────────────────────────────────────────────────
def filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb):
    """Keep only bitstrings with correct electron counts in each spin sector."""
    alpha_ok  = bsm[:, :n_orb].sum(axis=1) == n_alpha
    beta_ok   = bsm[:, n_orb:].sum(axis=1) == n_beta
    valid     = alpha_ok & beta_ok
    return bsm[valid], probs[valid]

# ─────────────────────────────────────────────────────────────────────────────
# 4. Build circuit (HF state + ansatz)
# ─────────────────────────────────────────────────────────────────────────────
hf_circuit = QuantumCircuit(n_qubits)
for i in range(n_alpha):        # alpha sector: qubits 0..n_orb-1
    hf_circuit.x(i)
for i in range(n_beta):         # beta sector:  qubits n_orb..2*n_orb-1
    hf_circuit.x(n_orb + i)

# ── Use function form (not deprecated class) ──────────────────────────────
ansatz = efficient_su2(
    n_qubits,
    reps                      = 2,
    entanglement              = "linear",
    skip_final_rotation_layer = True,
)

rng    = np.random.default_rng(42)
params = rng.uniform(-0.3, 0.3, ansatz.num_parameters)

circuit = hf_circuit.compose(ansatz.assign_parameters(params))
circuit.measure_all()
print(f"Circuit: {circuit.num_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Sample
# ─────────────────────────────────────────────────────────────────────────────
sampler = StatevectorSampler()
n_shots = 50_000      # more shots → more valid configs after filtering

counts = (
    sampler
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)
bitstring_matrix, probs = counts_to_arrays(counts)
print(f"Total unique bitstrings  : {bitstring_matrix.shape[0]}")

# ── FIX: pre-filter BEFORE the loop ──────────────────────────────────────
bitstring_matrix, probs = filter_bitstrings(
    bitstring_matrix, probs, n_alpha, n_beta, n_orb
)
print(f"Valid (correct e-count)  : {bitstring_matrix.shape[0]}")

if bitstring_matrix.shape[0] == 0:
    raise RuntimeError("No valid bitstrings — increase n_shots or adjust circuit params")

# ─────────────────────────────────────────────────────────────────────────────
# 6. SQD self-consistent loop
#    avg_occs shape: (2, n_orb)  — row 0 = alpha, row 1 = beta
#    Initialise with HF occupancies so recover_configurations
#    runs from iteration 0 onwards
# ─────────────────────────────────────────────────────────────────────────────
avg_occs          = np.zeros((2, n_orb))
avg_occs[0, :n_alpha] = 1.0    # alpha HF: fill lowest n_alpha orbitals
avg_occs[1, :n_beta]  = 1.0    # beta  HF: fill lowest n_beta  orbitals

n_iterations = 6
sqd_energy   = None

print(f"\nSQD iterations:")
print(f"{'─'*50}")

for it in range(n_iterations):

    # ── a) Configuration recovery ─────────────────────────────────────────
    bitstring_matrix, probs = recover_configurations(
        bitstring_matrix,
        probs,
        avg_occs,
        num_elec_a     = n_alpha,
        num_elec_b     = n_beta,
        num_iterations = 10,
        rand_seed      = 42,
    )

    if bitstring_matrix.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configurations after recovery")
        break

    # ── b) Solve H in bitstring subspace ─────────────────────────────────
    sqd_energy, coeffs, avg_occs = solve_fermion(
        bitstring_matrix,
        hcore      = h1e,
        eri        = h2e,
        open_shell = False,
        spin_sq    = 0.0,
    )

    print(f"  Iter {it+1:02d} | E = {sqd_energy:.8f} Ha | configs = {bitstring_matrix.shape[0]}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Final comparison
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*50}")
print(f"  DMET/FCI reference  : {dmet_energy:.8f} Ha")
print(f"  SQD result (frag 0) : {sqd_energy:.8f} Ha")
print(f"  Δ energy            : {abs(sqd_energy - dmet_energy):.2e} Ha")
print(f"{'═'*50}")

DMET/FCI reference energy = -2.13888991 Ha
Fragment 0 → 4 orbs | 2α + 2β electrons | 8 qubits
Circuit: 8 qubits | 15 depth | 32 params
Total unique bitstrings  : 56
Valid (correct e-count)  : 8

SQD iterations:
──────────────────────────────────────────────────


TypeError: recover_configurations() got an unexpected keyword argument 'num_iterations'

In [19]:
import inspect
from qiskit_addon_sqd.configuration_recovery import recover_configurations
from qiskit_addon_sqd.fermion import solve_fermion

# ── Check exact signature of your installed version ───────────────────────
print(inspect.signature(recover_configurations))
print(inspect.signature(solve_fermion))

(bitstring_matrix: 'np.ndarray', probabilities: 'Sequence[float] | np.ndarray', avg_occupancies: 'tuple[np.ndarray, np.ndarray]', num_elec_a: 'int', num_elec_b: 'int', rand_seed: 'np.random.Generator | int | None' = None) -> 'tuple[np.ndarray, np.ndarray]'
(bitstring_matrix: 'tuple[np.ndarray, np.ndarray] | np.ndarray', /, hcore: 'np.ndarray', eri: 'np.ndarray', *, open_shell: 'bool' = False, spin_sq: 'float | None' = None, shift: 'float' = 0.1, **kwargs) -> 'tuple[float, SCIState, tuple[np.ndarray, np.ndarray], float]'


In [20]:
import numpy as np
from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition

from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorSampler

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ─────────────────────────────────────────────────────────────────────────────
# 1. DMET
# ─────────────────────────────────────────────────────────────────────────────
geometry = [("H", (0., 0., i * 0.74)) for i in range(4)]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"DMET/FCI reference energy = {dmet_energy:.8f} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Extract fragment 0 integrals
# ─────────────────────────────────────────────────────────────────────────────
frag_data = dmet.scf_fragments[0]
h1e       = frag_data[1]
h2e       = frag_data[5]
n_alpha   = int(frag_data[3][0])
n_beta    = int(frag_data[3][1])
n_orb     = h1e.shape[0]
n_qubits  = 2 * n_orb

print(f"Fragment 0 → {n_orb} orbs | {n_alpha}α + {n_beta}β electrons | {n_qubits} qubits")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Filter helper
# ─────────────────────────────────────────────────────────────────────────────
def filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb):
    alpha_ok = bsm[:, :n_orb].sum(axis=1) == n_alpha
    beta_ok  = bsm[:, n_orb:].sum(axis=1) == n_beta
    valid    = alpha_ok & beta_ok
    return bsm[valid], probs[valid]

# ─────────────────────────────────────────────────────────────────────────────
# 4. Circuit
# ─────────────────────────────────────────────────────────────────────────────
hf_circuit = QuantumCircuit(n_qubits)
for i in range(n_alpha):
    hf_circuit.x(i)
for i in range(n_beta):
    hf_circuit.x(n_orb + i)

ansatz = efficient_su2(n_qubits, reps=2, entanglement="linear", skip_final_rotation_layer=True)
rng    = np.random.default_rng(42)
params = rng.uniform(-0.3, 0.3, ansatz.num_parameters)

circuit = hf_circuit.compose(ansatz.assign_parameters(params))
circuit.measure_all()
print(f"Circuit: {circuit.num_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Sample
# ─────────────────────────────────────────────────────────────────────────────
sampler = StatevectorSampler()
n_shots = 100_000     # more shots → more valid configs after filtering

counts = (
    sampler
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)
bitstring_matrix, probs = counts_to_arrays(counts)
print(f"Total unique bitstrings : {bitstring_matrix.shape[0]}")

bitstring_matrix, probs = filter_bitstrings(bitstring_matrix, probs, n_alpha, n_beta, n_orb)
print(f"Valid (correct e-count) : {bitstring_matrix.shape[0]}")

if bitstring_matrix.shape[0] == 0:
    raise RuntimeError("No valid bitstrings — increase n_shots")

# ─────────────────────────────────────────────────────────────────────────────
# 6. SQD loop
# ─────────────────────────────────────────────────────────────────────────────
avg_occs              = np.zeros((2, n_orb))
avg_occs[0, :n_alpha] = 1.0
avg_occs[1, :n_beta]  = 1.0

n_iterations = 6
sqd_energy   = None

# ── Print recover_configurations signature so we know what args to pass ───
print(f"\nrecover_configurations signature: {inspect.signature(recover_configurations)}")
print(f"solve_fermion signature         : {inspect.signature(solve_fermion)}")

print(f"\nSQD iterations:")
print(f"{'─'*50}")

for it in range(n_iterations):

    # ── a) Configuration recovery — only pass args in the signature ───────
    try:
        # newer API (no num_iterations)
        bitstring_matrix, probs = recover_configurations(
            bitstring_matrix,
            probs,
            avg_occs,
            num_elec_a = n_alpha,
            num_elec_b = n_beta,
            rand_seed  = 42,
        )
    except TypeError:
        # fallback: positional only
        bitstring_matrix, probs = recover_configurations(
            bitstring_matrix,
            probs,
            avg_occs,
            n_alpha,
            n_beta,
        )

    if bitstring_matrix.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configs after recovery")
        break

    # ── b) Solve fermion ──────────────────────────────────────────────────
    try:
        # newer API (no num_alpha/num_beta — inferred from bitstrings)
        sqd_energy, coeffs, avg_occs = solve_fermion(
            bitstring_matrix,
            hcore      = h1e,
            eri        = h2e,
            open_shell = False,
            spin_sq    = 0.0,
        )
    except TypeError:
        # older API (explicit num_alpha/num_beta)
        sqd_energy, coeffs, avg_occs = solve_fermion(
            bitstring_matrix,
            hcore      = h1e,
            eri        = h2e,
            num_alpha  = n_alpha,
            num_beta   = n_beta,
            open_shell = False,
            spin_sq    = 0.0,
        )

    print(f"  Iter {it+1:02d} | E = {sqd_energy:.8f} Ha | configs = {bitstring_matrix.shape[0]}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Final comparison
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*50}")
print(f"  DMET/FCI reference  : {dmet_energy:.8f} Ha")
print(f"  SQD result (frag 0) : {sqd_energy:.8f} Ha")
print(f"  Δ energy            : {abs(sqd_energy - dmet_energy):.2e} Ha")
print(f"{'═'*50}")

DMET/FCI reference energy = -2.13888991 Ha
Fragment 0 → 4 orbs | 2α + 2β electrons | 8 qubits
Circuit: 8 qubits | 15 depth | 32 params
Total unique bitstrings : 75
Valid (correct e-count) : 8

recover_configurations signature: (bitstring_matrix: 'np.ndarray', probabilities: 'Sequence[float] | np.ndarray', avg_occupancies: 'tuple[np.ndarray, np.ndarray]', num_elec_a: 'int', num_elec_b: 'int', rand_seed: 'np.random.Generator | int | None' = None) -> 'tuple[np.ndarray, np.ndarray]'
solve_fermion signature         : (bitstring_matrix: 'tuple[np.ndarray, np.ndarray] | np.ndarray', /, hcore: 'np.ndarray', eri: 'np.ndarray', *, open_shell: 'bool' = False, spin_sq: 'float | None' = None, shift: 'float' = 0.1, **kwargs) -> 'tuple[float, SCIState, tuple[np.ndarray, np.ndarray], float]'

SQD iterations:
──────────────────────────────────────────────────


ValueError: too many values to unpack (expected 3)

In [21]:
import inspect
import numpy as np
from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition

from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorSampler

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ─────────────────────────────────────────────────────────────────────────────
# 1. DMET
# ─────────────────────────────────────────────────────────────────────────────
geometry = [("H", (0., 0., i * 0.74)) for i in range(4)]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"DMET/FCI reference energy = {dmet_energy:.8f} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Extract fragment 0 integrals
# ─────────────────────────────────────────────────────────────────────────────
frag_data = dmet.scf_fragments[0]
h1e       = frag_data[1]
h2e       = frag_data[5]
n_alpha   = int(frag_data[3][0])    # 2
n_beta    = int(frag_data[3][1])    # 2
n_orb     = h1e.shape[0]            # 4
n_qubits  = 2 * n_orb               # 8

print(f"Fragment 0 → {n_orb} orbs | {n_alpha}α + {n_beta}β electrons | {n_qubits} qubits")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Filter helper — enforce correct electron count per spin sector
# ─────────────────────────────────────────────────────────────────────────────
def filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb):
    alpha_ok = bsm[:, :n_orb].sum(axis=1) == n_alpha
    beta_ok  = bsm[:, n_orb:].sum(axis=1) == n_beta
    valid    = alpha_ok & beta_ok
    return bsm[valid], probs[valid]

# ─────────────────────────────────────────────────────────────────────────────
# 4. Circuit: HF prep + EfficientSU2 ansatz
# ─────────────────────────────────────────────────────────────────────────────
hf_circuit = QuantumCircuit(n_qubits)
for i in range(n_alpha):       # alpha sector: qubits 0..n_orb-1
    hf_circuit.x(i)
for i in range(n_beta):        # beta  sector: qubits n_orb..2*n_orb-1
    hf_circuit.x(n_orb + i)

ansatz = efficient_su2(n_qubits, reps=2, entanglement="linear", skip_final_rotation_layer=True)
rng    = np.random.default_rng(42)
params = rng.uniform(-0.3, 0.3, ansatz.num_parameters)

circuit = hf_circuit.compose(ansatz.assign_parameters(params))
circuit.measure_all()
print(f"Circuit: {circuit.num_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Sample
# ─────────────────────────────────────────────────────────────────────────────
n_shots = 100_000
counts  = (
    StatevectorSampler()
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)

bitstring_matrix, probs = counts_to_arrays(counts)
print(f"Total unique bitstrings : {bitstring_matrix.shape[0]}")

bitstring_matrix, probs = filter_bitstrings(bitstring_matrix, probs, n_alpha, n_beta, n_orb)
print(f"Valid (correct e-count) : {bitstring_matrix.shape[0]}")

if bitstring_matrix.shape[0] == 0:
    raise RuntimeError("No valid bitstrings — increase n_shots")

# ─────────────────────────────────────────────────────────────────────────────
# 6. SQD loop
#
# KEY FIXES vs previous version:
#   ① solve_fermion returns 4 values: (energy, sci_state, avg_occs, spin_sq_val)
#      avg_occs is already tuple[np.ndarray, np.ndarray] → (alpha_occs, beta_occs)
#   ② recover_configurations expects avg_occupancies: tuple[np.ndarray, np.ndarray]
#      so initialise avg_occs as a tuple, not a (2, n_orb) array
# ─────────────────────────────────────────────────────────────────────────────

# Initialise with HF occupancies — tuple of (alpha_occs, beta_occs)
alpha_hf  = np.array([1.0 if i < n_alpha else 0.0 for i in range(n_orb)])
beta_hf   = np.array([1.0 if i < n_beta  else 0.0 for i in range(n_orb)])
avg_occs  = (alpha_hf, beta_hf)    # <── tuple[np.ndarray, np.ndarray]

n_iterations = 6
sqd_energy   = None
spin_sq_val  = None

print(f"\nSQD iterations:")
print(f"{'─'*55}")

for it in range(n_iterations):

    # ── a) Configuration recovery ─────────────────────────────────────────
    bitstring_matrix, probs = recover_configurations(
        bitstring_matrix,
        probs,
        avg_occs,           # tuple[ndarray, ndarray]  ← FIXED
        num_elec_a = n_alpha,
        num_elec_b = n_beta,
        rand_seed  = 42,
    )

    if bitstring_matrix.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configs after recovery")
        break

    # ── b) Solve H in bitstring subspace ─────────────────────────────────
    # Returns: (energy, SCIState, tuple[alpha_occs, beta_occs], spin_sq)
    sqd_energy, sci_state, avg_occs, spin_sq_val = solve_fermion(   # <── FIXED: 4 values
        bitstring_matrix,
        hcore      = h1e,
        eri        = h2e,
        open_shell = False,
        spin_sq    = 0.0,
    )

    print(f"  Iter {it+1:02d} | E = {sqd_energy:.8f} Ha | "
          f"configs = {bitstring_matrix.shape[0]} | "
          f"<S²> = {spin_sq_val:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Final comparison
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*55}")
print(f"  DMET/FCI reference  : {dmet_energy:.8f} Ha")
print(f"  SQD result (frag 0) : {sqd_energy:.8f} Ha")
print(f"  Δ energy            : {abs(sqd_energy - dmet_energy):.2e} Ha")
print(f"  Final <S²>          : {spin_sq_val:.6f}  (0 = singlet)")
print(f"{'═'*55}")

DMET/FCI reference energy = -2.13888991 Ha
Fragment 0 → 4 orbs | 2α + 2β electrons | 8 qubits
Circuit: 8 qubits | 15 depth | 32 params
Total unique bitstrings : 79
Valid (correct e-count) : 9

SQD iterations:
───────────────────────────────────────────────────────
  Iter 01 | E = -5.12845906 Ha | configs = 9 | <S²> = 0.0032
  Iter 02 | E = -5.12845906 Ha | configs = 9 | <S²> = 0.0032
  Iter 03 | E = -5.12845906 Ha | configs = 9 | <S²> = 0.0032
  Iter 04 | E = -5.12845906 Ha | configs = 9 | <S²> = 0.0032
  Iter 05 | E = -5.12845906 Ha | configs = 9 | <S²> = 0.0032
  Iter 06 | E = -5.12845906 Ha | configs = 9 | <S²> = 0.0032

═══════════════════════════════════════════════════════
  DMET/FCI reference  : -2.13888991 Ha
  SQD result (frag 0) : -5.12845906 Ha
  Δ energy            : 2.99e+00 Ha
  Final <S²>          : 0.003220  (0 = singlet)
═══════════════════════════════════════════════════════


In [22]:
import inspect
import numpy as np
from pyscf import fci as pyscf_fci

from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition

from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorSampler

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ─────────────────────────────────────────────────────────────────────────────
# 1. DMET
# ─────────────────────────────────────────────────────────────────────────────
geometry = [("H", (0., 0., i * 0.74)) for i in range(4)]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"DMET total energy = {dmet_energy:.8f} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Extract fragment 0
# ─────────────────────────────────────────────────────────────────────────────
frag_data = dmet.scf_fragments[0]
h1e       = frag_data[1]
h2e       = frag_data[5]
n_alpha   = int(frag_data[3][0])    # 2
n_beta    = int(frag_data[3][1])    # 2
n_orb     = h1e.shape[0]            # 4
n_qubits  = 2 * n_orb               # 8

print(f"Fragment 0 → {n_orb} orbs | {n_alpha}α + {n_beta}β | {n_qubits} qubits")
print(f"Max possible valid configs: C({n_orb},{n_alpha})² = {int(np.math.comb(n_orb,n_alpha)**2)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. FCI reference ON THE SAME FRAGMENT HAMILTONIAN  ← correct comparison
# ─────────────────────────────────────────────────────────────────────────────
cisolver   = pyscf_fci.direct_spin1.FCI()
n_elec     = (n_alpha, n_beta)
fci_energy, fci_vec = cisolver.kernel(h1e, h2e, n_orb, n_elec)
print(f"FCI fragment 0 energy     = {fci_energy:.8f} Ha  ← correct SQD target")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Filter helper
# ─────────────────────────────────────────────────────────────────────────────
def filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb):
    alpha_ok = bsm[:, :n_orb].sum(axis=1) == n_alpha
    beta_ok  = bsm[:, n_orb:].sum(axis=1) == n_beta
    valid    = alpha_ok & beta_ok
    return bsm[valid], probs[valid]

# ─────────────────────────────────────────────────────────────────────────────
# 5. Circuit — more reps for better coverage of the 36-config space
# ─────────────────────────────────────────────────────────────────────────────
hf_circuit = QuantumCircuit(n_qubits)
for i in range(n_alpha):
    hf_circuit.x(i)
for i in range(n_beta):
    hf_circuit.x(n_orb + i)

ansatz = efficient_su2(n_qubits, reps=3, entanglement="full", skip_final_rotation_layer=True)
rng    = np.random.default_rng(42)
params = rng.uniform(0, 2*np.pi, ansatz.num_parameters)   # full rotation range

circuit = hf_circuit.compose(ansatz.assign_parameters(params))
circuit.measure_all()
print(f"\nCircuit: {circuit.num_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Sample — need many shots to cover all 36 valid configurations
# ─────────────────────────────────────────────────────────────────────────────
n_shots = 500_000
counts  = (
    StatevectorSampler()
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)

bitstring_matrix, probs = counts_to_arrays(counts)
print(f"Total unique bitstrings : {bitstring_matrix.shape[0]}")

bitstring_matrix, probs = filter_bitstrings(bitstring_matrix, probs, n_alpha, n_beta, n_orb)
print(f"Valid (correct e-count) : {bitstring_matrix.shape[0]}")

if bitstring_matrix.shape[0] == 0:
    raise RuntimeError("No valid bitstrings — increase n_shots")

# ─────────────────────────────────────────────────────────────────────────────
# 7. SQD loop
# ─────────────────────────────────────────────────────────────────────────────
alpha_hf  = np.array([1.0 if i < n_alpha else 0.0 for i in range(n_orb)])
beta_hf   = np.array([1.0 if i < n_beta  else 0.0 for i in range(n_orb)])
avg_occs  = (alpha_hf, beta_hf)

n_iterations = 10
sqd_energy   = None
spin_sq_val  = None

print(f"\nSQD iterations (target FCI = {fci_energy:.8f} Ha):")
print(f"{'─'*65}")

for it in range(n_iterations):

    bitstring_matrix, probs = recover_configurations(
        bitstring_matrix,
        probs,
        avg_occs,
        num_elec_a = n_alpha,
        num_elec_b = n_beta,
        rand_seed  = 42,
    )

    if bitstring_matrix.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configs"); break

    sqd_energy, sci_state, avg_occs, spin_sq_val = solve_fermion(
        bitstring_matrix,
        hcore      = h1e,
        eri        = h2e,
        open_shell = False,
        spin_sq    = 0.0,
    )

    delta = abs(sqd_energy - fci_energy)
    print(f"  Iter {it+1:02d} | E = {sqd_energy:.8f} Ha | "
          f"configs = {bitstring_matrix.shape[0]:3d} | "
          f"<S²> = {spin_sq_val:.4f} | "
          f"ΔE from FCI = {delta:.2e} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Final comparison — all against THE SAME Hamiltonian
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*65}")
print(f"  Fragment 0 FCI energy   : {fci_energy:.8f} Ha  ← exact reference")
print(f"  Fragment 0 SQD energy   : {sqd_energy:.8f} Ha  ← SQD result")
print(f"  Δ (SQD vs FCI)          : {abs(sqd_energy - fci_energy):.2e} Ha")
print(f"  Final <S²>              : {spin_sq_val:.6f}  (0 = singlet ✓)")
print(f"\n  DMET total energy       : {dmet_energy:.8f} Ha  (different quantity)")
print(f"{'═'*65}")

DMET total energy = -2.13888991 Ha
Fragment 0 → 4 orbs | 2α + 2β | 8 qubits


AttributeError: module 'numpy' has no attribute 'math'

In [23]:
import inspect
import math                          # ← add this import
import numpy as np
from pyscf import fci as pyscf_fci

from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition

from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorSampler

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ─────────────────────────────────────────────────────────────────────────────
# 1. DMET
# ─────────────────────────────────────────────────────────────────────────────
geometry = [("H", (0., 0., i * 0.74)) for i in range(4)]
mol = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol,
    "fragment_atoms"  : [2, 2],
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"DMET total energy = {dmet_energy:.8f} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Extract fragment 0
# ─────────────────────────────────────────────────────────────────────────────
frag_data = dmet.scf_fragments[0]
h1e       = frag_data[1]
h2e       = frag_data[5]
n_alpha   = int(frag_data[3][0])
n_beta    = int(frag_data[3][1])
n_orb     = h1e.shape[0]
n_qubits  = 2 * n_orb

print(f"Fragment 0 → {n_orb} orbs | {n_alpha}α + {n_beta}β | {n_qubits} qubits")
print(f"Max possible valid configs: C({n_orb},{n_alpha})² = {math.comb(n_orb, n_alpha)**2}")   # ← FIXED

# ─────────────────────────────────────────────────────────────────────────────
# 3. FCI reference on the same fragment Hamiltonian
# ─────────────────────────────────────────────────────────────────────────────
cisolver   = pyscf_fci.direct_spin1.FCI()
n_elec     = (n_alpha, n_beta)
fci_energy, fci_vec = cisolver.kernel(h1e, h2e, n_orb, n_elec)
print(f"FCI fragment 0 energy     = {fci_energy:.8f} Ha  ← correct SQD target")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Filter helper
# ─────────────────────────────────────────────────────────────────────────────
def filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb):
    alpha_ok = bsm[:, :n_orb].sum(axis=1) == n_alpha
    beta_ok  = bsm[:, n_orb:].sum(axis=1) == n_beta
    valid    = alpha_ok & beta_ok
    return bsm[valid], probs[valid]

# ─────────────────────────────────────────────────────────────────────────────
# 5. Circuit
# ─────────────────────────────────────────────────────────────────────────────
hf_circuit = QuantumCircuit(n_qubits)
for i in range(n_alpha):
    hf_circuit.x(i)
for i in range(n_beta):
    hf_circuit.x(n_orb + i)

ansatz = efficient_su2(n_qubits, reps=3, entanglement="full", skip_final_rotation_layer=True)
rng    = np.random.default_rng(42)
params = rng.uniform(0, 2*np.pi, ansatz.num_parameters)

circuit = hf_circuit.compose(ansatz.assign_parameters(params))
circuit.measure_all()
print(f"\nCircuit: {circuit.num_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Sample
# ─────────────────────────────────────────────────────────────────────────────
n_shots = 500_000
counts  = (
    StatevectorSampler()
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)

bitstring_matrix, probs = counts_to_arrays(counts)
print(f"Total unique bitstrings : {bitstring_matrix.shape[0]}")

bitstring_matrix, probs = filter_bitstrings(bitstring_matrix, probs, n_alpha, n_beta, n_orb)
print(f"Valid (correct e-count) : {bitstring_matrix.shape[0]}")

if bitstring_matrix.shape[0] == 0:
    raise RuntimeError("No valid bitstrings — increase n_shots")

# ─────────────────────────────────────────────────────────────────────────────
# 7. SQD loop
# ─────────────────────────────────────────────────────────────────────────────
alpha_hf  = np.array([1.0 if i < n_alpha else 0.0 for i in range(n_orb)])
beta_hf   = np.array([1.0 if i < n_beta  else 0.0 for i in range(n_orb)])
avg_occs  = (alpha_hf, beta_hf)

n_iterations = 10
sqd_energy   = None
spin_sq_val  = None

print(f"\nSQD iterations (target FCI = {fci_energy:.8f} Ha):")
print(f"{'─'*65}")

for it in range(n_iterations):

    bitstring_matrix, probs = recover_configurations(
        bitstring_matrix,
        probs,
        avg_occs,
        num_elec_a = n_alpha,
        num_elec_b = n_beta,
        rand_seed  = 42,
    )

    if bitstring_matrix.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configs"); break

    sqd_energy, sci_state, avg_occs, spin_sq_val = solve_fermion(
        bitstring_matrix,
        hcore      = h1e,
        eri        = h2e,
        open_shell = False,
        spin_sq    = 0.0,
    )

    delta = abs(sqd_energy - fci_energy)
    print(f"  Iter {it+1:02d} | E = {sqd_energy:.8f} Ha | "
          f"configs = {bitstring_matrix.shape[0]:3d} | "
          f"<S²> = {spin_sq_val:.4f} | "
          f"ΔE from FCI = {delta:.2e} Ha")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Final summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*65}")
print(f"  Fragment 0 FCI energy   : {fci_energy:.8f} Ha  ← exact reference")
print(f"  Fragment 0 SQD energy   : {sqd_energy:.8f} Ha  ← SQD result")
print(f"  Δ (SQD vs FCI)          : {abs(sqd_energy - fci_energy):.2e} Ha")
print(f"  Final <S²>              : {spin_sq_val:.6f}  (0 = singlet ✓)")
print(f"\n  DMET total energy       : {dmet_energy:.8f} Ha  (different quantity)")
print(f"{'═'*65}")

DMET total energy = -2.13888991 Ha
Fragment 0 → 4 orbs | 2α + 2β | 8 qubits
Max possible valid configs: C(4,2)² = 36
FCI fragment 0 energy     = -5.23767538 Ha  ← correct SQD target

Circuit: 8 qubits | 37 depth | 48 params
Total unique bitstrings : 256
Valid (correct e-count) : 36

SQD iterations (target FCI = -5.23767538 Ha):
─────────────────────────────────────────────────────────────────
  Iter 01 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.0000 | ΔE from FCI = 4.87e-11 Ha
  Iter 02 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.0000 | ΔE from FCI = 4.87e-11 Ha
  Iter 03 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.0000 | ΔE from FCI = 4.87e-11 Ha
  Iter 04 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.0000 | ΔE from FCI = 4.87e-11 Ha
  Iter 05 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.0000 | ΔE from FCI = 4.87e-11 Ha
  Iter 06 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.0000 | ΔE from FCI = 4.87e-11 Ha
  Iter 07 | E = -5.23767538 Ha | configs =  36 | <S²> = 0.

In [25]:
import math
import numpy as np
from pyscf import gto, scf, mp, lo
from pyscf.tools import cubegen

# ─────────────────────────────────────────────────────────────────────────────
# Choose your molecule — swap this block freely
# ─────────────────────────────────────────────────────────────────────────────
MOLECULE = "H6"

geometries = {
    "LiH" : [("Li", (0., 0., 0.00)), ("H",  (0., 0., 1.60))],
    "H2O" : [("O",  (0., 0., 0.00)), ("H",  (0., 0.757,  0.586)), ("H",  (0., -0.757, 0.586))],
    "N2"  : [("N",  (0., 0., 0.00)), ("N",  (0., 0., 1.098))],
    "H6"  : [("H",  (0., 0., i*0.74)) for i in range(6)],
    "BeH2": [("Be", (0., 0., 0.00)), ("H",  (0., 0., 1.33)), ("H",  (0., 0., -1.33))],
}

geometry  = geometries[MOLECULE]
n_atoms   = len(geometry)
atom_syms = [a[0] for a in geometry]
print(f"Molecule : {MOLECULE}  ({n_atoms} atoms: {atom_syms})")

# ─────────────────────────────────────────────────────────────────────────────
# Step 1 — Mean-field (RHF) with PySCF
# ─────────────────────────────────────────────────────────────────────────────
mol_pyscf = gto.M(
    atom    = geometry,
    basis   = "sto-3g",
    charge  = 0,
    spin    = 0,
    verbose = 0,
)
mf = scf.RHF(mol_pyscf).run()
print(f"\nRHF energy = {mf.e_tot:.8f} Ha")
print(f"n_orbitals = {mol_pyscf.nao_nr()}")
print(f"n_electrons (alpha, beta) = {mol_pyscf.nelec}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 2 — MP2 Natural Orbital Occupation Numbers (NOONs)
#           NOONs far from 0 or 2 → strongly correlated orbitals
#           These are the ones that NEED to be in the active space
# ─────────────────────────────────────────────────────────────────────────────
mp2_calc      = mp.MP2(mf).run()
dm1_mp2       = mp2_calc.make_rdm1()                    # 1-RDM in MO basis
noons, natorbs = np.linalg.eigh(dm1_mp2)               # eigenvalues = NOONs
noons          = noons[::-1]                            # sort descending
natorbs        = natorbs[:, ::-1]

print(f"\nMP2 Natural Orbital Occupation Numbers (NOONs):")
print(f"{'Orbital':>8} {'NOON':>10} {'Character':>20}")
print("─" * 42)
for i, n in enumerate(noons):
    if n > 1.98:
        char = "core/doubly occupied"
    elif n > 0.02:
        char = "<<< ACTIVE >>>"         # far from 0 or 2
    else:
        char = "virtual"
    print(f"{i:>8d} {n:>10.6f} {char:>20}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — Select active orbitals automatically
#           Threshold: 0.02 < NOON < 1.98
# ─────────────────────────────────────────────────────────────────────────────
NOON_THRESHOLD_LOW  = 0.02
NOON_THRESHOLD_HIGH = 1.98

active_mask    = (noons > NOON_THRESHOLD_LOW) & (noons < NOON_THRESHOLD_HIGH)
active_indices = np.where(active_mask)[0]
n_active_orbs  = len(active_indices)
n_active_elecs = round(noons[active_mask].sum())    # electrons in active orbitals

print(f"\nActive space selection (threshold {NOON_THRESHOLD_LOW} < NOON < {NOON_THRESHOLD_HIGH}):")
print(f"  Active orbital indices : {active_indices.tolist()}")
print(f"  n_active_orbitals      : {n_active_orbs}")
print(f"  n_active_electrons     : {n_active_elecs}")

if n_active_orbs == 0:
    raise RuntimeError("No active orbitals found — molecule may not be correlated. Try N2 or H6.")

# ─────────────────────────────────────────────────────────────────────────────
# Step 4 — Localize active natural orbitals (Pipek-Mezey)
#           so we can assign them to atoms
# ─────────────────────────────────────────────────────────────────────────────

# Build MO coefficients for active natural orbitals only
active_natorb_coeffs = mf.mo_coeff @ natorbs[:, active_indices]   # AO → active NOs

# Pipek-Mezey localization
loc = lo.PM(mol_pyscf, active_natorb_coeffs)
loc.verbose = 0
loc_orbs = loc.kernel()    # shape: (n_AO, n_active_orbs)

print(f"\nLocalized active orbital coefficients shape: {loc_orbs.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 5 — Mulliken population analysis
#           → assign each localized orbital to an atom
#           → tells us WHICH ATOMS are most correlated
# ─────────────────────────────────────────────────────────────────────────────
S         = mf.get_ovlp()        # AO overlap matrix
ao_labels = mol_pyscf.ao_labels(fmt=None)   # list of (atom_idx, symbol, ao_type, ...)

# For each localized active orbital, compute |C_μ|² * S_μμ per atom
orbital_atom_weight = np.zeros((n_active_orbs, n_atoms))

for orb_i in range(n_active_orbs):
    c = loc_orbs[:, orb_i]
    # Mulliken charge: q_A = Σ_{μ∈A} (C·S·C)_μ
    CS = c * (S @ c)
    for ao_j, (atom_idx, *_) in enumerate(ao_labels):
        orbital_atom_weight[orb_i, atom_idx] += CS[ao_j]

print(f"\nOrbital-to-atom Mulliken weights (rows=active orbitals, cols=atoms):")
header = "Orb\\Atom" + "".join(f"{s:>10}" for s in atom_syms)
print(header)
for i, row in enumerate(orbital_atom_weight):
    vals = "".join(f"{v:>10.4f}" for v in row)
    assigned = int(np.argmax(row))
    print(f"  Orb {active_indices[i]:2d}  {vals}  → atom {assigned} ({atom_syms[assigned]})")

# ─────────────────────────────────────────────────────────────────────────────
# Step 6 — Assign each active orbital to its dominant atom
#           → count active orbitals per atom
#           → find most correlated atoms (quantum fragment candidates)
# ─────────────────────────────────────────────────────────────────────────────
dominant_atoms = np.argmax(orbital_atom_weight, axis=1)   # atom index per orbital
active_per_atom = np.bincount(dominant_atoms, minlength=n_atoms)

print(f"\nActive orbital count per atom:")
for i, (sym, count) in enumerate(zip(atom_syms, active_per_atom)):
    bar = "█" * int(count * 5)
    print(f"  Atom {i} ({sym:2s}) : {count:2d} active orbitals  {bar}")

most_correlated_atom = int(np.argmax(active_per_atom))
print(f"\n→ Most correlated atom: {most_correlated_atom} ({atom_syms[most_correlated_atom]})")

Molecule : H6  (6 atoms: ['H', 'H', 'H', 'H', 'H', 'H'])

RHF energy = -3.07994194 Ha
n_orbitals = 6
n_electrons (alpha, beta) = (3, 3)

MP2 Natural Orbital Occupation Numbers (NOONs):
 Orbital       NOON            Character
──────────────────────────────────────────
       0   1.994735 core/doubly occupied
       1   1.989645 core/doubly occupied
       2   1.979953       <<< ACTIVE >>>
       3   0.023433       <<< ACTIVE >>>
       4   0.008957              virtual
       5   0.003276              virtual

Active space selection (threshold 0.02 < NOON < 1.98):
  Active orbital indices : [2, 3]
  n_active_orbitals      : 2
  n_active_electrons     : 2

Localized active orbital coefficients shape: (6, 2)

Orbital-to-atom Mulliken weights (rows=active orbitals, cols=atoms):
Orb\Atom         H         H         H         H         H         H
  Orb  2      0.2509    0.0407    0.2084    0.2084    0.0407    0.2509  → atom 5 (H)
  Orb  3      0.2473    0.0609    0.1917    0.1917    0.0609

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# HQS ActiveSpaceFinder (commercial/research tool by HQS Quantum Simulations)
# Install: pip install hqs-active-space-finder   (check HQS docs for exact name)
# Docs   : https://www.hqs.srl/
#
# The HQS ASF uses DMRG orbital entanglement entropy to select active spaces
# This is MORE accurate than NOON thresholds for strongly correlated systems
# ─────────────────────────────────────────────────────────────────────────────

try:
    # Attempt HQS ActiveSpaceFinder import
    # NOTE: exact package/class name — check your HQS installation
    from hqs_active_space_finder import ActiveSpaceFinder    # adjust import as needed

    asf = ActiveSpaceFinder(
        mol          = mol_pyscf,
        mo_coeff     = mf.mo_coeff,
        mo_occ       = mf.mo_occ,
        max_orbitals = 8,            # max orbitals in active space
        max_electrons= 8,            # max electrons in active space
    )
    asf_result      = asf.run()
    active_indices  = asf_result.active_orbital_indices
    n_active_orbs   = len(active_indices)
    n_active_elecs  = asf_result.n_active_electrons

    print(f"HQS ASF active orbitals  : {active_indices}")
    print(f"HQS ASF n_active_orbs    : {n_active_orbs}")
    print(f"HQS ASF n_active_electrons: {n_active_elecs}")
    HQS_AVAILABLE = True

except ImportError:
    print("HQS ActiveSpaceFinder not found — using MP2 NOON selection instead")
    HQS_AVAILABLE = False
    # active_indices already set from Step 3 above

HQS ActiveSpaceFinder not found — using MP2 NOON selection instead


In [27]:
from asf import wrapper

In [35]:
import math
import numpy as np
from pyscf import gto, fci as pyscf_fci

# ── Correct imports from docs ─────────────────────────────────────────────────
from asf.wrapper import runasf_from_mole     # takes Mole object, does SCF internally
# from asf.wrapper import runasf_from_scf   # alternative: pass your own SCF object

# ── Molecule ──────────────────────────────────────────────────────────────────
MOLECULE = "H2O"

geometries = {
    "LiH" : [("Li", (0., 0., 0.00)), ("H",  (0., 0., 1.60))],
    "H2O" : [("O",  (0., 0., 0.00)), ("H",  (0.,  0.757,  0.586)),
                                      ("H",  (0., -0.757,  0.586))],
    "N2"  : [("N",  (0., 0., 0.00)), ("N",  (0., 0., 1.098))],
    "H6"  : [("H",  (0., 0., i*0.74)) for i in range(6)],
}

geometry  = geometries[MOLECULE]
atom_syms = [a[0] for a in geometry]
n_atoms   = len(geometry)

# ── PySCF Mole ────────────────────────────────────────────────────────────────
# verbose=3 (INFO) → shows entropy table from docs (w(), S, sel columns)
mol_pyscf = gto.M(
    atom    = geometry,
    basis   = "sto-3g",
    charge  = 0,
    spin    = 0,
    verbose = 3,
)

# ── ASF ───────────────────────────────────────────────────────────────────────
# From docs, runasf_from_mole internally:
#   1) UHF + stability analysis (intentional symmetry breaking)
#   2) MP2 natural orbitals
#   3) CASCI or DMRG-CI to compute one-orbital entropies S
#   4) Select orbitals where S > entropy_threshold
#
# Returns (from docs):
#   nel      : int         number of electrons in selected active space
#   mo_list  : list[int]   active orbital indices, 0-based, in MP2 NO basis
#   mo_coeff : ndarray     (n_AO, n_MO) MP2 natural orbital coefficients
#                          NOTE: mo_list refers to THIS matrix, NOT mf.mo_coeff

nel, mo_list, mo_coeff = runasf_from_mole(
    mol_pyscf,
    entropy_threshold = 0.15,   # default ~0.139; lower → more orbitals selected
                                # higher → fewer orbitals selected
)

# ── Alternative: use your own SCF ─────────────────────────────────────────────
# from pyscf import scf
# from asf.wrapper import runasf_from_scf
# mf = scf.RHF(mol_pyscf).run()
# nel, mo_list, mo_coeff = runasf_from_scf(mf, entropy_threshold=0.15)

print(f"\n{'='*55}")
print(f"ASF Active Space Result:")
print(f"  n_active_electrons : {nel}")
print(f"  n_active_orbitals  : {len(mo_list)}")
print(f"  orbital indices    : {mo_list}  (MP2 NO basis, 0-indexed)")
print(f"{'='*55}")

if len(mo_list) == 0:
    raise RuntimeError(
        "ASF found no active orbitals.\n"
        "Try: lower entropy_threshold (e.g. 0.05) or choose a more correlated molecule (N2, H6)"
    )

# ── Map active orbitals → atoms via Mulliken population ───────────────────────
# mo_coeff[:, mo_list] → (n_AO, n_active) coefficients of active NOs
# Mulliken weight of orbital i on atom A: Σ_{μ∈A} C_μi * (S·C)_μi

S          = mol_pyscf.intor('int1e_ovlp')        # AO overlap matrix (n_AO, n_AO)
ao_labels  = mol_pyscf.ao_labels(fmt=None)         # [(atom_idx, sym, ao_name, ...), ...]

active_coeffs       = mo_coeff[:, mo_list]         # (n_AO, n_active_orbs)
orbital_atom_weight = np.zeros((len(mo_list), n_atoms))

for orb_i in range(len(mo_list)):
    c  = active_coeffs[:, orb_i]
    CS = c * (S @ c)                               # Mulliken product per AO
    for ao_j, (atom_idx, *_) in enumerate(ao_labels):
        orbital_atom_weight[orb_i, atom_idx] += CS[ao_j]

dominant_atoms  = np.argmax(orbital_atom_weight, axis=1)   # most-weighted atom per orbital
active_per_atom = np.bincount(dominant_atoms, minlength=n_atoms)

print("\nOrbital → Atom mapping (Mulliken):")
print(f"  {'NO idx':>6}  {'→ atom':>8}  {'Weights per atom'}")
print(f"  {'─'*50}")
for i, (orb_idx, da) in enumerate(zip(mo_list, dominant_atoms)):
    weights = "  ".join(f"{atom_syms[j]}:{orbital_atom_weight[i,j]:+.3f}" for j in range(n_atoms))
    print(f"  NO {orb_idx:3d}  → atom {da} ({atom_syms[da]:2s})  |  {weights}")

print(f"\nActive orbital count per atom:")
for i, (sym, cnt) in enumerate(zip(atom_syms, active_per_atom)):
    bar = "█" * int(cnt * 4)
    print(f"  Atom {i} ({sym:2s}): {cnt:2d} active orbitals  {bar}")

most_active_frag = int(np.argmax(active_per_atom))
print(f"\n→ Most correlated fragment (ASF-guided): "
      f"atom {most_active_frag} ({atom_syms[most_active_frag]})")

ImportError: cannot import name 'runasf_from_mole' from 'asf.wrapper' (/home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/lib/python3.11/site-packages/asf/wrapper.py)

In [33]:
import asf
import asf.wrapper

print(f"ASF Version : {getattr(asf,'__version__','Unknown')}")
print(f"ASF Location: {asf.__file__}")

print(dir(asf.wrapper))

from pyscf import dmrgscf
print(f"DMRG executable config : {dmrgscf.settings.BLOCKEXE}")

ASF Version : Unknown
ASF Location: /home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/lib/python3.11/site-packages/asf/__init__.py
['ASFBase', 'ASFCI', 'ASFDMRG', 'ActiveSpace', 'Any', 'DEFAULT_ENTROPY_THRESHOLD', 'DEFAULT_MAX_RESTARTS', 'DEFAULT_SCF_SETTINGS', 'FilterFunction', 'MOListInfo', 'MP2NatorbPreselection', 'Mole', 'Optional', 'SCFclass', 'Union', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calc_ncore', 'create_asf_switched_cisolver', 'find_from_mol', 'find_from_scf', 'inactive_orbital_lists', 'loghead', 'loginfo', 'merge_active_spaces', 'np', 'print_mo_table', 'reorder_mos', 'sized_space_from_mol', 'sized_space_from_scf', 'stable_scf']
DMRG executable config : /home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/bin/block2main


In [37]:
import math
import numpy as np
from pyscf import gto, scf, fci as pyscf_fci, lo

# ── Correct imports from wrapper.py ──────────────────────────────────────────
from asf.wrapper import (
    find_from_mol,          # Mole → ActiveSpace (entropy threshold)
    find_from_scf,          # SCF  → ActiveSpace (entropy threshold)
    sized_space_from_mol,   # Mole → ActiveSpace (exact size)
    sized_space_from_scf,   # SCF  → ActiveSpace (exact size)
    reorder_mos,            # utility: sort cols as occ/active/virt
)

# ── Molecule ──────────────────────────────────────────────────────────────────
MOLECULE = "H6"

geometries = {
    "LiH" : [("Li", (0., 0., 0.00)), ("H",  (0., 0., 1.60))],
    "H2O" : [("O",  (0., 0., 0.00)), ("H",  (0.,  0.757,  0.586)),
                                      ("H",  (0., -0.757,  0.586))],
    "N2"  : [("N",  (0., 0., 0.00)), ("N",  (0., 0., 1.098))],
    "H6"  : [("H",  (0., 0., i*0.74)) for i in range(6)],
}

geometry  = geometries[MOLECULE]
atom_syms = [a[0] for a in geometry]
n_atoms   = len(geometry)

# ── PySCF Mole ────────────────────────────────────────────────────────────────
mol_pyscf = gto.M(
    atom    = geometry,
    basis   = "sto-3g",
    charge  = 0,
    spin    = 0,
    verbose = 3,            # INFO level → prints entropy table from ASF
)

# ─────────────────────────────────────────────────────────────────────────────
# Option A: find_from_mol — ASF does UHF + stability analysis internally
#           Best for general use, follows exact same procedure as in docs
# ─────────────────────────────────────────────────────────────────────────────
active_space = find_from_mol(
    mol_pyscf,
    entropy_threshold = 0.15,   # orbitals with S > threshold are selected
                                # lower  → more orbitals selected
                                # higher → fewer orbitals selected
    max_norb          = 8,      # cap active space size (optional)
    min_norb          = 2,      # minimum active space size (optional)
    verbose           = True,   # prints entropy table
)

# ─────────────────────────────────────────────────────────────────────────────
# Option B: find_from_scf — bring your own RHF/UHF object
#           Use this if you want control over the SCF step
# ─────────────────────────────────────────────────────────────────────────────
# mf = scf.RHF(mol_pyscf).run()
# active_space = find_from_scf(
#     mf,
#     entropy_threshold = 0.15,
#     states            = 1,    # ground state only
#                               # states=2 → ground + first excited
#                               # states=[(0,1),(2,1)] → 1 singlet + 1 triplet
#     verbose           = True,
# )

# ─────────────────────────────────────────────────────────────────────────────
# Option C: sized_space_from_mol — request exact active space size
#           Use when you know exactly how many orbitals you want
# ─────────────────────────────────────────────────────────────────────────────
# active_space = sized_space_from_mol(
#     mol_pyscf,
#     size    = (2, 2),   # (n_electrons, n_orbitals) e.g. (2,2) = HOMO+LUMO
#                         # or just size=4 → 4 orbitals, electrons inferred
#     verbose = True,
# )

# ── Unpack ActiveSpace object ─────────────────────────────────────────────────
# Returns ActiveSpace dataclass with .nel, .mo_list, .mo_coeff
nel      = active_space.nel        # int: n_active_electrons
mo_list  = active_space.mo_list    # list[int]: active orbital indices (0-based, MP2 NO basis)
mo_coeff = active_space.mo_coeff   # ndarray (n_AO, n_MO): MP2 natural orbital coefficients

n_active_orbs = len(mo_list)

print(f"\n{'='*55}")
print(f"ASF Results for {MOLECULE}:")
print(f"  n_active_electrons : {nel}")
print(f"  n_active_orbitals  : {n_active_orbs}")
print(f"  active mo_list     : {mo_list}  (0-indexed, MP2 NO basis)")
print(f"  mo_coeff shape     : {mo_coeff.shape}")
print(f"{'='*55}")

if n_active_orbs == 0:
    raise RuntimeError(
        "ASF found no active orbitals.\n"
        "Try: lower entropy_threshold (e.g. 0.05) "
        "or use N2/H6 for stronger correlation"
    )

# ── Map active orbitals → atoms via Mulliken population ───────────────────────
# mo_coeff[:, mo_list] → coefficients of active NOs only
# mo_list indices refer to columns of mo_coeff (MP2 NO basis)

S         = mol_pyscf.intor("int1e_ovlp")      # AO overlap (n_AO, n_AO)
ao_labels = mol_pyscf.ao_labels(fmt=None)       # [(atom_idx, sym, ao_name, ...), ...]

active_coeffs       = mo_coeff[:, mo_list]      # (n_AO, n_active_orbs)
orbital_atom_weight = np.zeros((n_active_orbs, n_atoms))

for orb_i in range(n_active_orbs):
    c  = active_coeffs[:, orb_i]
    CS = c * (S @ c)                            # Mulliken product per AO
    for ao_j, (atom_idx, *_) in enumerate(ao_labels):
        orbital_atom_weight[orb_i, atom_idx] += CS[ao_j]

dominant_atoms  = np.argmax(orbital_atom_weight, axis=1)
active_per_atom = np.bincount(dominant_atoms, minlength=n_atoms)

print("\nOrbital → Atom Mulliken mapping:")
print(f"  {'MO':>4}  {'→ atom':>7}  {'Weights per atom'}")
print(f"  {'─'*52}")
for i, (orb_idx, da) in enumerate(zip(mo_list, dominant_atoms)):
    weights = "  ".join(
        f"{atom_syms[j]}:{orbital_atom_weight[i,j]:+.3f}" for j in range(n_atoms)
    )
    print(f"  {orb_idx:4d}  → {da} ({atom_syms[da]:2s})  |  {weights}")

print(f"\nActive orbital count per atom:")
for i, (sym, cnt) in enumerate(zip(atom_syms, active_per_atom)):
    bar = "█" * int(cnt * 4)
    print(f"  Atom {i} ({sym:2s}): {cnt:2d}  {bar}")

most_active_frag = int(np.argmax(active_per_atom))
print(f"\n→ ASF-guided fragment: atom {most_active_frag} ({atom_syms[most_active_frag]})")


--------------------------------------------------------------------------------
Calculating UHF orbitals
--------------------------------------------------------------------------------

-> Initiating a UHF calculation.
converged SCF energy = -3.07994194122782  <S^2> = 4.4408921e-15  2S+1 = 1
<class 'pyscf.scf.uhf.UHF'> wavefunction is stable in the internal stability analysis
-> The calculated SCF solution is converged and stable.

--------------------------------------------------------------------------------
Calculating MP2 natural orbitals
--------------------------------------------------------------------------------


-> Selected initial orbital window of 6 electrons in 6 MP2 natural orbitals.

--------------------------------------------------------------------------------
Running calculation
--------------------------------------------------------------------------------

spin = 0, number of roots = 1
total number of roots calculated = 1

CASCI E = -3.14236549875558  E(CI) 

In [38]:
from tangelo import SecondQuantizedMolecule
from tangelo.problem_decomposition import DMETProblemDecomposition
from qiskit import QuantumCircuit
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorSampler
from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ── DMET fragment definition ──────────────────────────────────────────────────
# One fragment per atom (safest, most general)
# ASF tells us WHICH fragment is most correlated → send that one to SQD
fragment_atoms = [1] * n_atoms

print(f"Fragment definition : {fragment_atoms}  (1 per atom)")
print(f"SQD target fragment : {most_active_frag} ({atom_syms[most_active_frag]})"
      f"  ← ASF-guided")

# ── DMET ──────────────────────────────────────────────────────────────────────
mol_tangelo = SecondQuantizedMolecule(geometry, q=0, spin=0, basis="sto-3g")

dmet = DMETProblemDecomposition({
    "molecule"        : mol_tangelo,
    "fragment_atoms"  : fragment_atoms,
    "fragment_solvers": "fci",
    "verbose"         : False,
})
dmet.build()
dmet_energy = dmet.simulate()
print(f"\nDMET total energy = {dmet_energy:.8f} Ha")

# ── Extract ASF-guided fragment integrals ─────────────────────────────────────
# scf_fragments[i] = [RHF, h1e, Mole, [n_a,n_b], fock, h2e, fock_copy]
#                     [0]   [1]  [2]   [3]        [4]   [5]  [6]

frag_data = dmet.scf_fragments[most_active_frag]
h1e       = frag_data[1]               # embedded one-electron integrals
h2e       = frag_data[5]               # embedded two-electron integrals
n_alpha   = int(frag_data[3][0])
n_beta    = int(frag_data[3][1])
n_orb     = h1e.shape[0]
n_qubits  = 2 * n_orb

print(f"\nFragment {most_active_frag} ({atom_syms[most_active_frag]}): "
      f"{n_orb} orbs | {n_alpha}α+{n_beta}β | {n_qubits} qubits")
print(f"Max valid configs = C({n_orb},{n_alpha})² = {math.comb(n_orb, n_alpha)**2}")

# ── FCI reference on the same fragment Hamiltonian ────────────────────────────
cisolver  = pyscf_fci.direct_spin1.FCI()
fci_energy, _ = cisolver.kernel(h1e, h2e, n_orb, (n_alpha, n_beta))
print(f"FCI fragment energy = {fci_energy:.8f} Ha  ← correct SQD target")

# ── Filter helper ─────────────────────────────────────────────────────────────
def filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb):
    valid = (
        (bsm[:, :n_orb].sum(axis=1) == n_alpha) &
        (bsm[:, n_orb:].sum(axis=1) == n_beta)
    )
    return bsm[valid], probs[valid]

# ── Circuit ───────────────────────────────────────────────────────────────────
hf_circ = QuantumCircuit(n_qubits)
for i in range(n_alpha): hf_circ.x(i)
for i in range(n_beta):  hf_circ.x(n_orb + i)

ansatz  = efficient_su2(
    n_qubits,
    reps                      = 3,
    entanglement              = "full",
    skip_final_rotation_layer = True,
)
rng     = np.random.default_rng(42)
params  = rng.uniform(0, 2*np.pi, ansatz.num_parameters)
circuit = hf_circ.compose(ansatz.assign_parameters(params))
circuit.measure_all()
print(f"\nCircuit: {n_qubits} qubits | {circuit.depth()} depth | {ansatz.num_parameters} params")

# ── Sample ────────────────────────────────────────────────────────────────────
n_shots = 500_000
counts  = (
    StatevectorSampler()
    .run([circuit], shots=n_shots)
    .result()[0]
    .data.meas
    .get_counts()
)
bsm, probs = counts_to_arrays(counts)
bsm, probs = filter_bitstrings(bsm, probs, n_alpha, n_beta, n_orb)
print(f"Valid bitstrings: {bsm.shape[0]} / {math.comb(n_orb, n_alpha)**2} max")

if bsm.shape[0] == 0:
    raise RuntimeError("No valid bitstrings — increase n_shots")

# ── SQD loop ──────────────────────────────────────────────────────────────────
avg_occs = (
    np.array([1.0 if i < n_alpha else 0.0 for i in range(n_orb)]),
    np.array([1.0 if i < n_beta  else 0.0 for i in range(n_orb)]),
)
sqd_energy  = None
spin_sq_val = None

print(f"\nSQD iterations (FCI target = {fci_energy:.8f} Ha):")
print("─" * 68)

for it in range(10):
    bsm, probs = recover_configurations(
        bsm, probs, avg_occs,
        num_elec_a = n_alpha,
        num_elec_b = n_beta,
        rand_seed  = 42,
    )
    if bsm.shape[0] == 0:
        print(f"  [iter {it+1}] No valid configs after recovery"); break

    sqd_energy, _, avg_occs, spin_sq_val = solve_fermion(
        bsm,
        hcore      = h1e,
        eri        = h2e,
        open_shell = False,
        spin_sq    = 0.0,
    )
    print(f"  Iter {it+1:02d} | E={sqd_energy:.8f} Ha | "
          f"configs={bsm.shape[0]:3d} | "
          f"<S²>={spin_sq_val:.4f} | "
          f"ΔE={abs(sqd_energy-fci_energy):.2e} Ha")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═'*68}")
print(f"  Molecule                          : {MOLECULE}")
print(f"  ASF: {nel}e in {n_active_orbs} orbs → orbitals {mo_list}")
print(f"  ASF-guided fragment               : {most_active_frag} ({atom_syms[most_active_frag]})")
print(f"  DMET fragments                    : {fragment_atoms}")
print(f"  FCI fragment {most_active_frag} energy         : {fci_energy:.8f} Ha")
print(f"  SQD fragment {most_active_frag} energy         : {sqd_energy:.8f} Ha")
print(f"  Δ (SQD vs FCI)                    : {abs(sqd_energy-fci_energy):.2e} Ha")
print(f"  Final <S²>                        : {spin_sq_val:.6f}  (0=singlet ✓)")
print(f"  DMET total energy                 : {dmet_energy:.8f} Ha")
print(f"{'═'*68}")

Fragment definition : [1, 1, 1, 1, 1, 1]  (1 per atom)
SQD target fragment : 5 (H)  ← ASF-guided

DMET total energy = -3.13674985 Ha

Fragment 5 (H): 2 orbs | 1α+1β | 4 qubits
Max valid configs = C(2,1)² = 4
FCI fragment energy = -1.66149642 Ha  ← correct SQD target

Circuit: 4 qubits | 21 depth | 24 params
Valid bitstrings: 4 / 4 max

SQD iterations (FCI target = -1.66149642 Ha):
────────────────────────────────────────────────────────────────────
  Iter 01 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 02 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 03 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 04 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 05 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 06 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 07 | E=-1.66149642 Ha | configs=  4 | <S²>=0.0000 | ΔE=1.78e-15 Ha
  Iter 08 | E=-1.66149642 Ha 